In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

In [ ]:
# Last IPL match ingested into gold.duckdb
q(f"""
SELECT m.match_id, m.match_date, m.season, m.team_1, m.team_2,
    coalesce(m.event_stage, 'League')        AS stage,
    coalesce(m.outcome_winner, 'No result')  AS result
FROM main_marts.fct_match_results m
WHERE m.event_name ILIKE '%indian premier league%'
ORDER BY m.match_date DESC LIMIT 1
""")

In [ ]:
# ── Set team and season here ─────────────────────────────────────────────────
TEAM   = "Sunrisers Hyderabad"
SEASON = '2026'

# For teams that changed official name add alternate names here.
# Example for RCB: TEAM_ALIASES = ["Royal Challengers Bengaluru"]
TEAM_ALIASES = []
# ─────────────────────────────────────────────────────────────────────────────

_names  = [TEAM] + TEAM_ALIASES
IN_LIST = ", ".join(f"'{t}'" for t in _names)
print(f"Team: {TEAM}  |  Season: {SEASON}")

## Season Record

In [ ]:
# Match-by-match results for the season
q(f"""
SELECT
    m.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END  AS vs,
    coalesce(m.event_stage, 'League')                                   AS stage,
    CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 'W'
         WHEN m.is_no_result THEN 'NR'
         WHEN m.is_tie       THEN 'T'
         ELSE 'L' END                                                   AS result,
    m.outcome_by_runs                                                   AS by_runs,
    m.outcome_by_wickets                                                AS by_wkts,
    m.outcome_method
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
ORDER BY m.match_date
""")

In [ ]:
# Season summary — W/L/NR
q(f"""
SELECT
    count(*)                                                                    AS played,
    sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END)            AS won,
    sum(CASE WHEN m.has_result AND m.outcome_winner NOT IN ({IN_LIST})
         THEN 1 ELSE 0 END)                                                     AS lost,
    sum(m.is_tie::integer)                                                      AS tied,
    sum(m.is_no_result::integer)                                                AS nr,
    round(sum(CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 1 ELSE 0 END) * 100.0
          / nullif(sum(m.has_result::integer), 0), 1)                           AS win_pct
FROM main_marts.fct_match_results m
WHERE (m.team_1 IN ({IN_LIST}) OR m.team_2 IN ({IN_LIST}))
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
""")

## All Teams — Top 3 Batting Concentration

In [ ]:
# All teams — % of runs scored by top 3 batters, ordered by concentration
q(f"""
WITH batter_runs AS (
    SELECT
        b.batting_team                                      AS team,
        b.player_name                                       AS batter,
        sum(b.runs)                                         AS runs
    FROM main_intermediate.int_batting_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
    GROUP BY b.batting_team, b.player_name
),
ranked AS (
    SELECT team, batter, runs,
        row_number() OVER (PARTITION BY team ORDER BY runs DESC) AS rnk
    FROM batter_runs
),
team_totals AS (
    SELECT team, sum(runs) AS total_runs
    FROM batter_runs
    GROUP BY team
),
top3 AS (
    SELECT team,
        sum(runs)                                                                       AS top3_runs,
        string_agg(batter || ' (' || runs::integer || ')', ', ' ORDER BY runs DESC)    AS top3_players
    FROM ranked
    WHERE rnk <= 3
    GROUP BY team
)
SELECT
    t.team,
    t.total_runs,
    x.top3_runs,
    round(x.top3_runs * 100.0 / t.total_runs, 1)   AS top3_pct,
    x.top3_players
FROM team_totals t
JOIN top3 x ON t.team = x.team
ORDER BY top3_pct DESC
""")

## Batting Contributions

In [ ]:
# All batters — season batting summary (super overs excluded)
q(f"""
SELECT
    b.player_name                                                               AS batter,
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    sum(b.not_out::integer)                                                     AS no,
    sum(b.runs)                                                                 AS runs,
    max(b.runs)                                                                 AS hs,
    round(sum(b.runs)::double / nullif(sum(b.dismissals), 0), 2)                AS avg,
    sum(b.balls_faced)                                                          AS bf,
    round(sum(b.runs) * 100.0 / nullif(sum(b.balls_faced), 0), 2)               AS sr,
    sum(CASE WHEN b.runs >= 50 AND b.runs < 100 THEN 1 ELSE 0 END)              AS "50s",
    sum(CASE WHEN b.runs >= 100 THEN 1 ELSE 0 END)                              AS "100s",
    sum(b.fours)                                                                AS "4s",
    sum(b.sixes)                                                                AS "6s"
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY runs DESC
""")

In [ ]:
# Top 3 batters — share of team runs this season
q(f"""
WITH team_runs AS (
    SELECT sum(b.runs) AS total_runs
    FROM main_intermediate.int_batting_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE b.batting_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
)
SELECT
    b.player_name                                                              AS batter,
    sum(b.runs)                                                                AS runs,
    round(sum(b.runs) * 100.0 / (SELECT total_runs FROM team_runs), 1)        AS pct_of_team_runs
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY runs DESC
LIMIT 3
""")

## Bowling Contributions

In [ ]:
# All bowlers — season bowling summary (super overs excluded)
q(f"""
SELECT
    b.player_name                                                               AS bowler,
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    (floor(sum(b.legal_deliveries) / 6)::integer
     || '.' || (sum(b.legal_deliveries) % 6))::varchar                         AS overs,
    sum(b.legal_deliveries)                                                     AS balls,
    sum(b.runs_conceded)                                                        AS runs,
    sum(b.wickets_credited)                                                     AS wkts,
    round(sum(b.runs_conceded)::double / nullif(sum(b.wickets_credited), 0), 2) AS avg,
    round(sum(b.runs_conceded) * 6.0 / nullif(sum(b.legal_deliveries), 0), 2)  AS econ,
    round(sum(b.legal_deliveries)::double
          / nullif(sum(b.wickets_credited), 0), 2)                              AS sr,
    sum(CASE WHEN b.wickets_credited >= 4 THEN 1 ELSE 0 END)                   AS "4w+"
FROM main_intermediate.int_bowling_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY wkts DESC
""")

In [ ]:
# Top 3 bowlers — share of team wickets this season
q(f"""
WITH team_wkts AS (
    SELECT sum(b.wickets_credited) AS total_wkts
    FROM main_intermediate.int_bowling_by_innings b
    JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
    WHERE b.bowling_team IN ({IN_LIST})
      AND m.event_name ILIKE '%indian premier league%'
      AND m.season = '{SEASON}'
      AND b.innings_number IN (1, 2)
      AND NOT b.super_over
)
SELECT
    b.player_name                                                                   AS bowler,
    sum(b.wickets_credited)                                                         AS wkts,
    round(sum(b.wickets_credited) * 100.0 / (SELECT total_wkts FROM team_wkts), 1) AS pct_of_team_wkts
FROM main_intermediate.int_bowling_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
GROUP BY b.player_name
ORDER BY wkts DESC
LIMIT 3
""")

## Player Deep-Dive

In [ ]:
# ── Set player name here for match-by-match breakdown ────────────────────────
PLAYER = "Abhishek Sharma"
# ─────────────────────────────────────────────────────────────────────────────
print(f"Deep-dive: {PLAYER}  |  {TEAM}  |  {SEASON}")

In [ ]:
# Match-by-match batting for selected player this season
q(f"""
SELECT
    b.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END  AS vs,
    CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 'W'
         WHEN m.is_no_result THEN 'NR'
         WHEN m.is_tie       THEN 'T'
         ELSE 'L' END                                                   AS result,
    b.runs,
    b.not_out,
    b.balls_faced                                                        AS balls,
    round(b.runs * 100.0 / nullif(b.balls_faced, 0), 1)                 AS sr,
    b.fours  AS "4s",
    b.sixes  AS "6s"
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.player_name = '{PLAYER}'
  AND b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
ORDER BY b.match_date
""")

In [ ]:
# Season batting summary for selected player
q(f"""
SELECT
    count(DISTINCT b.match_id)                                                  AS mat,
    count(*)                                                                    AS inns,
    sum(b.not_out::integer)                                                     AS no,
    sum(b.runs)                                                                 AS runs,
    max(b.runs)                                                                 AS hs,
    round(sum(b.runs)::double / nullif(sum(b.dismissals), 0), 2)                AS avg,
    sum(b.balls_faced)                                                          AS bf,
    round(sum(b.runs) * 100.0 / nullif(sum(b.balls_faced), 0), 2)               AS sr,
    sum(CASE WHEN b.runs >= 50 AND b.runs < 100 THEN 1 ELSE 0 END)              AS "50s",
    sum(CASE WHEN b.runs >= 100 THEN 1 ELSE 0 END)                              AS "100s",
    sum(b.fours)                                                                AS "4s",
    sum(b.sixes)                                                                AS "6s"
FROM main_intermediate.int_batting_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.player_name = '{PLAYER}'
  AND b.batting_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
""")

In [ ]:
# Match-by-match bowling for selected player this season
q(f"""
SELECT
    b.match_date,
    CASE WHEN m.team_1 IN ({IN_LIST}) THEN m.team_2 ELSE m.team_1 END       AS vs,
    CASE WHEN m.outcome_winner IN ({IN_LIST}) THEN 'W'
         WHEN m.is_no_result THEN 'NR'
         WHEN m.is_tie       THEN 'T'
         ELSE 'L' END                                                        AS result,
    (floor(b.legal_deliveries / 6)::integer
     || '.' || (b.legal_deliveries % 6))::varchar                            AS overs,
    b.legal_deliveries                                                        AS balls,
    b.runs_conceded                                                           AS runs,
    b.wickets_credited                                                        AS wkts,
    round(b.runs_conceded * 6.0 / nullif(b.legal_deliveries, 0), 2)          AS econ
FROM main_intermediate.int_bowling_by_innings b
JOIN main_marts.fct_match_results m ON b.match_id = m.match_id
WHERE b.player_name = '{PLAYER}'
  AND b.bowling_team IN ({IN_LIST})
  AND m.event_name ILIKE '%indian premier league%'
  AND m.season = '{SEASON}'
  AND b.innings_number IN (1, 2)
  AND NOT b.super_over
ORDER BY b.match_date
""")